In [1]:
!pip install great-expectations==0.18.21

In [4]:
!pip uninstall -y numpy pandas great-expectations
!pip install numpy==1.26.4 pandas==2.2.2 great-expectations==0.18.21

Found existing installation: numpy 2.5.1
Uninstalling numpy-2.5.1:
  Successfully uninstalled numpy-2.5.1
Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: great-expectations 0.18.21
Uninstalling great-expectations-0.18.21:
  Successfully uninstalled great-expectations-0.18.21
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 35.9 MB/s eta 0:00:00
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you

In [1]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks/AI_Recruitment"

!pip install -q deltalake
from deltalake import DeltaTable

silver = DeltaTable(f"{BASE}/delta/silver_applications").to_pandas()
print(" Number of Row:", len(silver))
silver.head(3)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Number of Row: 33


,job_id,applicant_id,name,education,applicant_experience,skills,title,company,city,salary,required_skills,required_experience,application_id
0,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1
1,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1
2,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1


In [2]:
from openlineage.client import OpenLineageClient
from openlineage.client.transport.console import ConsoleTransport, ConsoleConfig
from openlineage.client.run import RunEvent, RunState, Run, Job
from openlineage.client.uuid import generate_new_uuid
from datetime import datetime, timezone
import json, os

NAMESPACE = "ai_recruitment"
PRODUCER  = "https://github.com/IsmailElbery/ai-recruitment"

client = OpenLineageClient(transport=ConsoleTransport(ConsoleConfig()))
lineage_log = []

def emit(job_name, state, run_id=None):
    run_id = run_id or str(generate_new_uuid())
    event = RunEvent(
        eventType=getattr(RunState, state),
        eventTime=datetime.now(timezone.utc).isoformat(),
        run=Run(runId=run_id),
        job=Job(namespace=NAMESPACE, name=job_name),
        producer=PRODUCER,
        inputs=[], outputs=[]
    )
    client.emit(event)
    lineage_log.append({"job": job_name, "state": state,
                        "run_id": run_id, "time": event.eventTime})
    print(f"[OpenLineage] {job_name} → {state}")
    return run_id

print("OpenLineage Ready")


OpenLineage Ready


In [3]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("AI Recruitment Lakehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [6]:
!pip install delta-spark==3.2.0 pyspark==3.5.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.1 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=5885f350b1084b0de369a0966a70f57a1d2531f3a05840949dbffda2b5fb0ced
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.3
    Uninstalling pyspark-4.0.3:
      Successfully uninstalled pyspark-4.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [10]:
from pyspark.sql.functions import col

print("Null applicant_id:", silver["applicant_id"].isna().sum())
print("Null job_id:", silver["job_id"].isna().sum())

print("Duplicate application_id:",
      silver["application_id"].duplicated().sum())

print("Invalid applicant experience:",
      ((silver["applicant_experience"] < 0) |
       (silver["applicant_experience"] > 50)).sum())

print("Invalid salary:",
      ((silver["salary"] <= 0) |
       (silver["salary"] > 100000)).sum())

Null applicant_id: 0
Null job_id: 0
Duplicate application_id: 24
Invalid applicant experience: 0
Invalid salary: 0


In [6]:
print(type(silver))

<class 'pandas.core.frame.DataFrame'>


In [7]:
print(col)

<function col at 0x7909598fc220>


In [13]:
def run_quality_checks(df, label="silver"):
    results = {
        "Null applicant_id": df["applicant_id"].isna().sum() == 0,
        "Null job_id": df["job_id"].isna().sum() == 0,
        "Applicant experience":
            ((df["applicant_experience"] >= 0) &
             (df["applicant_experience"] <= 50)).all(),
        "Salary":
            ((df["salary"] > 0) &
             (df["salary"] <= 100000)).all(),
        "Unique application_id":
            not df["application_id"].duplicated().any()
    }

    print(f"\n=== Quality Check Results ({label}) ===")

    for check, ok in results.items():
        print(f"{'✓' if ok else '✗'} {check}")

    passed = all(results.values())
    print(f"\nFinal result: {'Passed' if passed else 'Failed'}")

    return passed, results

In [14]:
bad = silver.copy()
bad.loc[bad.index[:3], "experience_years"] = -99
bad.loc[bad.index[3:5], "salary"] = 0

rid_bad = emit("quality_gate_silver_BAD", "START")
passed_bad, _ = run_quality_checks(bad, "Bad data ")

if passed_bad:
    emit("quality_gate_silver_BAD", "COMPLETE", rid_bad)
else:
    emit("quality_gate_silver_BAD", "FAIL", rid_bad)
    print("The gate has closed — subsequent stages were not executed")
    print(" gold_aggregation: SKIPPED")

[OpenLineage] quality_gate_silver_BAD → START

=== Quality Check Results (Bad data ) ===
✓ Null applicant_id
✓ Null job_id
✓ Applicant experience
✗ Salary
✓ Unique application_id

Final result: Failed
[OpenLineage] quality_gate_silver_BAD → FAIL
The gate has closed — subsequent stages were not executed
 gold_aggregation: SKIPPED


In [15]:
lineage_df = pd.DataFrame(lineage_log)
print(lineage_df.to_string(index=False))

os.makedirs(f"{BASE}/output", exist_ok=True)
lineage_df.to_json(f"{BASE}/output/lineage_events.json",
                   orient="records", force_ascii=False, indent=2)

print("\Summary:")
print(lineage_df["state"].value_counts().to_string())


<>:8: SyntaxWarning: invalid escape sequence '\S'
<>:8: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_4319/1718598161.py:8: SyntaxWarning: invalid escape sequence '\S'
  print("\Summary:")


                    job state                               run_id                             time
    quality_gate_silver START 019f8b83-6011-7b68-bcc4-15a365cd7a63 2026-07-22T20:27:50.674064+00:00
quality_gate_silver_BAD START 019f8b83-ea3b-7fb8-98d7-d1e9c69116cb 2026-07-22T20:28:26.043319+00:00
quality_gate_silver_BAD  FAIL 019f8b83-ea3b-7fb8-98d7-d1e9c69116cb 2026-07-22T20:28:26.048841+00:00
\Summary:
state
START    2
FAIL     1
